# Inventory Optimization: Predicting Dead Stock Proability


## Author: Ambika Gole

Strategic Objective: To identify stagnant inventory across Australian wearhouse facilities to optimise storage planning and free up capital.


Background: 
A  mid manufacturing retailer identified that specific products across Australian warehouses had zero sales activity over several months. This project utilizes classification algorithms to predict items likely to be classified as "Dead Stock" (inventory unlikely to be sold due to obsolescence or lack of demand).


Key Results:

- Best Performing Model: Decision Tree Classifier (96% Test Accuracy).

- Primary Drivers: Inventory Turn, % of Over 2 Year Stock, and Total Quantity.



## 1. Data Ingestion and Exploratory Analysis ##

In [25]:
#importing libraries
import pandas as pd
import numpy as np

#funtion to read excel file and return DataFrame
def create_df_excel(filepath, nrows= None):
    return pd.read_excel(filepath, nrows= nrows)

#loading the dataset 
df= create_df_excel('Dataset/Inventory.xlsx')

#checking columns names
print(df.columns.tolist())

#clean column names
df.columns = df.columns.str.strip()

#kepping first 3000 rows and dropping'Item No' column
df= df.head(3000).drop(columns=['Item No.'])

#printing dataframe information
df.info()

#displaying first 5 rows
df.head()

#removing duplicate 
df = df.drop_duplicates()

#checking the shape of the dataset
print('After dropping duplicates:', df.shape)

#After reporting, dropping Item Description (non-descriptive variables)
df = df.drop(columns=['Item Description'], errors='ignore')

#printing shape after dropping item description
print('Shape of dataset after dropping Item Description:', df.shape)

['Item No. ', 'Item Description', 'Whse', 'State', 'Base Unit', 'Total - Quantity', 'Inventory Aging Report Unit Cost', 'Total - Value', '6 Months QTY', '12 Months QTY', '2 Years QTY ', 'Over 2 Years Qty ', 'Over 3 Years Quantity ', 'Business Area', 'Item Type', 'ABC Class', 'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec', 'Avg monthly', 'Inventory Turn', 'Over 2 years Qty', '% of over 2 year', 'Dead stock']
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 3000 entries, 0 to 2999
Data columns (total 32 columns):
 #   Column                            Non-Null Count  Dtype  
---  ------                            --------------  -----  
 0   Item Description                  2999 non-null   object 
 1   Whse                              3000 non-null   object 
 2   State                             3000 non-null   object 
 3   Base Unit                         3000 non-null   object 
 4   Total - Quantity                  3000 non-null   float64
 5   

**Analysis:** 
I would recommend to drop columns like `Item Description` as it is just text description and not useful for analysis do not contribute in predicting dead stock.

|Variable Kind|Number of Features|Feature Names
| --- | --- | --- |
| **Numeric**| 24| Total-Quantity, Inventory Aging Report Unit Cost, Total-value, 6 Months QTY, 12 Months QTY, 2 Years QTY, Over 2 Years Qty, Over 3                    Years Quantity,  Jan–Dec, Avg monthly, Inventory Turn, % of over 2                     year |
| **Nominal**  | 6 | Whse, State, Base Unit, Business Area, Item Type, Dead Stock |
| **Ordinal**  | 1 | ABC Class |

**Description of table:**
The Inventory dataset contains 
- There are 24 **numeric variables**, which can be counted or measured such as stock quantities, costs, turnover and figures reflecting montly usage
- Likewise, there are 6 **nominal variables**, used to label varibales without any quantitative value and cannot be ordered in a meanigful way, like  Whse, State, Base Unit, Business Area, Item Type, Dead Stock
-  Lastly, there is one **ordinal variable** which is ABC class as it can be logically ordered or ranked along a scale like A= Critically Important, E = End of Life.

## 2. Data Quality Assessment ##

In [26]:
# Check missing values for each column
print("Missing values per column:\n")
print(df.isnull().sum())

# Print total missing values in the dataset
print(f"\nThere are {df.isnull().sum().sum()} missing values in the entire dataset.")

Missing values per column:

Whse                                0
State                               0
Base Unit                           0
Total - Quantity                    0
Inventory Aging Report Unit Cost    0
Total - Value                       0
6 Months QTY                        0
12 Months QTY                       0
2 Years QTY                         0
Over 2 Years Qty                    0
Over 3 Years Quantity               0
Business Area                       0
Item Type                           0
ABC Class                           0
Jan                                 0
Feb                                 0
Mar                                 0
Apr                                 0
May                                 0
Jun                                 0
Jul                                 0
Aug                                 0
Sep                                 0
Oct                                 0
Nov                                 0
Dec                   


**Description:** The Inventory dataset after keeping only 3000 first dataset is a complete dataset, with no missing value and it will not affect the predictive and modelling process.

In [27]:
##Dealing missing values
#Imputing numeric columns with mean
num_cols = df.select_dtypes(include = ['float64']).columns
df[num_cols] = df[num_cols].fillna(df[num_cols].mean())

#Imputing categorical columns with mode
cat_cols = df.select_dtypes(include=['object']).columns
for col in cat_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

#checking for null values
print('Total missing values after imputation:', df.isnull().sum().sum())

Total missing values after imputation: 0


**Explanation:** 
Data imputation means replacing missing values so that we do  not ahve drop any values from rows and column, which decrease the possibility of possible biased results.

The above result shows that it does not contain any missing values. However, I have shown the imputation process, such as for numerical columns I have used mean and mode for categorical columns. Due to no missing vlaues, the imputation result of missing value is 0.

In [28]:
#Printing value counts of 'State' column
print(df["State"].value_counts())

# Creating dummy column for 'NSW'
df["State_NSW"] = (df["State"] == "NSW").astype(int)

# Drop original column
df.drop("State", axis=1, inplace=True)

#checking for columns after dropping State
print(df.columns)

State
NSW    2113
WA      887
Name: count, dtype: int64
Index(['Whse', 'Base Unit', 'Total - Quantity',
       'Inventory Aging Report Unit Cost', 'Total - Value', '6 Months QTY',
       '12 Months QTY', '2 Years QTY', 'Over 2 Years Qty',
       'Over 3 Years Quantity', 'Business Area', 'Item Type', 'ABC Class',
       'Jan', 'Feb', 'Mar', 'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct',
       'Nov', 'Dec', 'Avg monthly', 'Inventory Turn', 'Over 2 years Qty',
       '% of over 2 year', 'Dead stock', 'State_NSW'],
      dtype='object')


**Explanation:** 
The new variable `State_NSW` has value 1 indicates the stock location is New South Wales(NSW) and value 0 if the stock location is another state, which in this dataset is WA.

In [29]:
#Value counts for Whse
print('Value counts for Whse:\n')
print(df['Whse'].value_counts())

Value counts for Whse:

Whse
1N1    1318
1W0     887
1N0     795
Name: count, dtype: int64


**Explanation:** The `Whse` variable gives us the wearhouse or store location of where the items are stored. There are three distintive codes, they are 1N1(1318 records), 1W0(887 records) and 1N0(795 records). This proves that this variable is categorical in nature. As per the distribution, 1N1 is the most frequently reported location. 

## 3. Preprocessing and Feature Engineering ##

In [30]:
# checking Whse values
df['Whse'] = df['Whse'].astype(str)
print(df['Whse'].value_counts())

# creating dummy variables
dummies = pd.get_dummies(df['Whse'])

#converting True and False to 1 and 0
dummies = dummies.astype(int)

#renaming columns to required names
dummies = dummies.rename(columns={'1N0': 'Whse_1N0',
                                  '1N1': 'Whse_1N1',
                                  '1W0': 'Whse_1W0'})

# adding dummies to df and drop original Whse
df = pd.concat([df.drop(columns=['Whse']), dummies], axis=1)

# verifying the dummy variable
print(df[['Whse_1N0','Whse_1N1','Whse_1W0']].head())



Whse
1N1    1318
1W0     887
1N0     795
Name: count, dtype: int64
   Whse_1N0  Whse_1N1  Whse_1W0
0         1         0         0
1         0         1         0
2         0         0         1
3         0         1         0
4         0         1         0


**Explanation:**
- The `Whse` variable has three categories, they are 1N0, 1N1 and 1W0
- For these outcome I have applied one-hot encoding to create new dummy variables, Whse_1N0, Whse_1N1 and Whse_1W0
- Likewise, these dummy columns have result 1 and 0, which means 1= True(belongs to that particular wearhouse) and 0= False(does not belong to that particular wearhouse)
- In this way no observation were deleted and no anomalies were assumed as missing
- Lastly I dropped the original `Whse` column to aviod duplication

In [31]:
# normalising values
s = df['Item Type'].astype(str).str.upper().str.strip()

# starting with 'Other' as default
df['Item_Type_4'] = 'Other'

# overwrite using .loc conditions
df.loc[s.str.startswith('FG'), 'Item_Type_4'] = 'Finished Goods'
df.loc[s.str.startswith('RM'), 'Item_Type_4'] = 'Raw Materials'
df.loc[s.str.startswith('WIP'), 'Item_Type_4'] = 'WIP Manufactured'

# checking result
print(df['Item_Type_4'].value_counts())

#checking 
print(df.select_dtypes(include=['object','bool']).columns)

##Encoding for non-numeric features

# Ordinal encoding for ABC Class
df['ABC Class'] = df['ABC Class'].map({'A': 3, 'B': 2, 'C': 1})

# Binary encoding for Dead stock
df['Dead stock'] = df['Dead stock'].map({'Yes': 1, 'No': 0})

# One-hot encoding nominal features
df = pd.get_dummies(df, columns=['Business Area', 'Base Unit', 'Item_Type_4'], drop_first=False)

# Checking remaining non-numeric
print(df.select_dtypes(include=['object','bool']).columns)


Item_Type_4
Finished Goods      2026
Raw Materials        761
WIP Manufactured     194
Other                 19
Name: count, dtype: int64
Index(['Base Unit', 'Business Area', 'Item Type', 'ABC Class', 'Dead stock',
       'Item_Type_4'],
      dtype='object')
Index(['Item Type', 'Business Area_0', 'Business Area_COM',
       'Business Area_DLT', 'Business Area_EXL', 'Business Area_FLD',
       'Business Area_HLB', 'Business Area_LCP', 'Business Area_LMP',
       'Business Area_PCT', 'Business Area_PEN', 'Business Area_RWY',
       'Business Area_SAE', 'Business Area_SCS', 'Business Area_SUR',
       'Business Area_TAL', 'Business Area_TRO', 'Business Area_URB',
       'Business Area_UVC', 'Base Unit_EA', 'Base Unit_KG', 'Base Unit_ML',
       'Base Unit_MT', 'Base Unit_PC', 'Item_Type_4_Finished Goods',
       'Item_Type_4_Other', 'Item_Type_4_Raw Materials',
       'Item_Type_4_WIP Manufactured'],
      dtype='object')


## 4. Feature Selection and Data Splitting ##

In [32]:

#cutoff for 80%
cutoff = int(len(df) * 0.8)

#creating y (Dead stock column only)
y = df['Dead stock'].iloc[:cutoff].values   

#creating X (all other variables except Dead stock)
X = df.drop(columns=['Dead stock']).iloc[:cutoff].values

#checking shape
print("y shape:", y.shape)
print("X shape:", X.shape)

#checking columns
df.columns

y shape: (2400,)
X shape: (2400, 57)


Index(['Total - Quantity', 'Inventory Aging Report Unit Cost', 'Total - Value',
       '6 Months QTY', '12 Months QTY', '2 Years QTY', 'Over 2 Years Qty',
       'Over 3 Years Quantity', 'Item Type', 'ABC Class', 'Jan', 'Feb', 'Mar',
       'Apr', 'May', 'Jun', 'Jul', 'Aug', 'Sep', 'Oct', 'Nov', 'Dec',
       'Avg monthly', 'Inventory Turn', 'Over 2 years Qty', '% of over 2 year',
       'Dead stock', 'State_NSW', 'Whse_1N0', 'Whse_1N1', 'Whse_1W0',
       'Business Area_0', 'Business Area_COM', 'Business Area_DLT',
       'Business Area_EXL', 'Business Area_FLD', 'Business Area_HLB',
       'Business Area_LCP', 'Business Area_LMP', 'Business Area_PCT',
       'Business Area_PEN', 'Business Area_RWY', 'Business Area_SAE',
       'Business Area_SCS', 'Business Area_SUR', 'Business Area_TAL',
       'Business Area_TRO', 'Business Area_URB', 'Business Area_UVC',
       'Base Unit_EA', 'Base Unit_KG', 'Base Unit_ML', 'Base Unit_MT',
       'Base Unit_PC', 'Item_Type_4_Finished Goods', 'Ite

In [33]:
#checking if there is any non-numeric column
print(df.select_dtypes(include=['object']).columns)

#One-hot encoding for Item Type
df = pd.get_dummies(df, columns=['Item Type'])

# Check again if all variables numeric or not
print(df.select_dtypes(include=['object']).columns)

# target
y = df['Dead stock'].values

# features (drop target)
X = df.drop(columns=['Dead stock']).values

print(X.dtype)

#importing library
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

# making sure all dummies are numeric (0 or 1, not bool)
df = df.astype(int, errors='ignore')

# building target and features
y = df['Dead stock'].astype(int).values
X = df.drop(columns=['Dead stock']).values

#checking datatypes for int64 or float64
print("X dtype before split:", X.dtype)   

# spliting into train(80) and test(20) 
X_train, X_test, y_train, y_test = train_test_split(
    X, y, test_size=0.2, random_state=31, stratify=y
)

# standardising (fit on train only, apply to both)
scaler = StandardScaler()
X_train = scaler.fit_transform(X_train)
X_test  = scaler.transform(X_test)

#printing shape 
print("X_train shape:", X_train.shape)
print("y_train shape:", y_train.shape)

Index(['Item Type'], dtype='object')
Index([], dtype='object')
object
X dtype before split: float64
X_train shape: (2400, 71)
y_train shape: (2400,)


## 5. Model Training and Performance Evaluation ##

In [34]:
#importing necessary libraries
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score

#Imputing missing values (mean for numeric, most_frequent for categorical)
imputer = SimpleImputer(strategy="mean")
X_train = imputer.fit_transform(X_train)
X_test  = imputer.transform(X_test)

# Training linear classifier (Logistic Regression)
clf = LogisticRegression(random_state=31, max_iter=1000)
clf.fit(X_train, y_train)

# Predictions
y_train_pred = clf.predict(X_train)
y_test_pred  = clf.predict(X_test)

# Computing accuracies
train_acc = accuracy_score(y_train, y_train_pred)
test_acc  = accuracy_score(y_test, y_test_pred)

#printing result
print("Training Accuracy:", train_acc)
print("Test Accuracy:", test_acc)

Training Accuracy: 0.9625
Test Accuracy: 0.9433333333333334


In [35]:

#importing libraries
from sklearn.tree import DecisionTreeClassifier
from sklearn.metrics import accuracy_score

# Training non-linear classifier (Decision Tree)
dt = DecisionTreeClassifier(random_state=31)
dt.fit(X_train, y_train)

# Predictions
y_train_pred = dt.predict(X_train)
y_test_pred  = dt.predict(X_test)

# Computing accuracies
train_acc = accuracy_score(y_train, y_train_pred)
test_acc  = accuracy_score(y_test, y_test_pred)

#printing result
print("Training Accuracy (Decision Tree):", train_acc)
print("Test Accuracy (Decision Tree):", test_acc)


Training Accuracy (Decision Tree): 1.0
Test Accuracy (Decision Tree): 0.96


## 6. Strategic Recommendations and Business Insights ##

### Analysis for Logistic Regression: ###
- Logistic Regression used for linear classification
- Training result ≈ 0.9625(96.3%) and Testing result ≈ 0.9433(94.3%), shows that the model generalises well, has good balance and low overfitting
- The model has stable performance across train and test sets
- The model might have missed complex and non-linear relationships in the data

### Analysis for Decision Tree: ###
- Decision Tree used for non-linear classfication
- Training result accuracy is 100% and testing result accuracy is 96%, shows that perfectly training score suggest overfItting of the model (model memorises training data well)
- The model has higher testing accuracy(96%) than logistic regression(94.3%)
- The model captures non-linear patterns and interactions between features

### Analysis of training and testing results: ###
- Logistic regression shows balanced train-test result, shows only small gap between training and testing results than Decision trees's train-test results, indicating strong generalisation
- Decision Tree has priominent gap in train and test scores than logistic regression, indicating mild overfitting despite high test accuracy
- Thus, Logistic Regression = less variance and Decision Tress = more variance but still good test result

### Model Comparision and Recommendations: ###
- Decision Tree recommended as it achieved high test accuarcy (96%) in comparision to Logistic Regression (94.3%)
- Decision Tree captures non-linear relationship which Logistic regression might have missed
- Decision Tree can find complex pattern in data and gives clear rules for conditional situations, so that managers can easily understand and use the result

### Features impacting the prediction of outcome variable: ###
- `% of over 2 year`: This feature shows the stock has been there for a while and is very old
- `Over 2 years Qty` : This feature shows that large amount of old stock is sitting and is a strong indicator of dead stock
- `Inventory Turn` : This feature shows that stocks are selling very slowly each month
- `Total-Quantity` : This feature shows that a high holding quantity carries the risk that some stock wont sell